# 03 收益风险与新闻情绪分类

## 本课学习目标

- A. 量化金融主线：收益率（Return）、波动率（Volatility）和最大回撤（Maximum Drawdown，MDD）
零基础解释：它们分别描述赚亏比例、波动大小和最大下跌。
- B. 大语言模型主线：情绪分析（Sentiment Analysis）和模拟提供商（Mock Provider）
零基础解释：Mock Provider 离线按关键词分类，不是真正的大语言模型。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：计算收益风险指标，用 MockLLMProvider 分类新闻并画图。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 可运行实验

下面代码只读取 `learning/data` 下的合成数据。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from learning.src.market_data import load_price_data
from learning.src.financial_metrics import simple_returns, maximum_drawdown, annualized_volatility
from learning.src.sentiment_factor import classify_news
prices = load_price_data(DATA / "sample_prices.csv")
aaa = prices[prices.ticker == "AAA"].set_index("date")["close"]
rets = simple_returns(aaa)
print("AAA annualized volatility:", round(annualized_volatility(rets), 4))
print("AAA max drawdown:", round(maximum_drawdown((1+rets).cumprod()), 4))
news = pd.read_csv(DATA / "sample_news.csv").head(15)
sent = classify_news(news)
display(sent[["ticker", "label", "score", "confidence", "reason"]].head())
sent["score"].hist()
plt.title("Mock sentiment scores")
plt.show()

## 结尾总结

你现在应该理解：收益、风险和情绪分类是量化策略的三大基础模块。

本课核心收获：
- 简单收益率和对数收益率描述价格变化比例；
- 年化波动率衡量收益的不确定性；
- 最大回撤从净值峰点到谷底计算最大累计亏损；
- Mock Provider 按关键词规则离线分类新闻情绪，不调用真实模型；
- Pydantic 约束 score 在 [-1,1]、confidence 在 [0,1]。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：
- Simple Return（简单收益率）
- Log Return（对数收益率）
- Annualized Volatility（年化波动率）
- Maximum Drawdown / MDD（最大回撤）
- Sentiment Analysis（情绪分析）
- Mock Provider（模拟提供商）

### 常见错误

1. **混淆简单收益率和对数收益率**：简单收益率 = (P₁−P₀)/P₀，对数收益率 = ln(P₁/P₀)，两者在小波动时接近但公式不同。
2. **用价格代替收益率计算波动率**：波动率应在收益率序列上计算，不能直接用价格。
3. **最大回撤只看跌幅不看顺序**：回撤必须是净值先创新高再下跌的最大幅度，不能随便取区间最低点。
4. **忘记年化**：日收益率的标准差乘以 √252 才是年化波动率，直接报告日数据会严重低估风险。

### 课后练习

1. **手动验证收益率**：取 AAA 某两天的收盘价，手工计算简单收益率和对数收益率，与 `simple_returns()` 函数输出对比。
2. **定位最大回撤**：找出 AAA 净值曲线的历史最高点日期和之后的最低点日期，确认回撤值等于 `maximum_drawdown()` 的输出。
3. **观察情绪分数**：将 MockLLMProvider 的正向关键词增加一个中文词（如"增长"），重新分类，观察 score 是否变化。

下一课与本课有什么关系：下一课将引入回测时间轴概念，讲解新闻时间对齐和未来数据泄漏的防范。